In [2]:
import os

In [3]:
%pwd

'd:\\Revision\\End-to-End-ML-project\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'd:\\Revision\\End-to-End-ML-project'

In [6]:
import dagshub
dagshub.init(repo_owner='shu7620', repo_name='End-to-End-ML-project', mlflow=True)

Accessing as shu7620

Initialized MLflow to track repo "shu7620/End-to-End-ML-project"

Repository shu7620/End-to-End-ML-project initialized!

In [23]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [24]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [25]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
        
    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema =  self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path = config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri="https://dagshub.com/shu7620/End-to-End-ML-project.mlflow",
           
        )

        return model_evaluation_config

In [26]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [27]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
        
    def eval_metrics(self,actual,pred):
        rmse=np.sqrt(mean_squared_error(actual,pred))
        mae=mean_absolute_error(actual,pred)
        r2=r2_score(actual,pred)
        return rmse,mae,r2
    
    def log_into_mlflow(self):
        test_data=pd.read_csv(self.config.test_data_path)
        model=joblib.load(self.config.model_path)
        
        test_x=test_data.drop([self.config.target_column],axis=1)
        test_y=test_data[[self.config.target_column]]
        
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            
            predicted_qualities=model.predict(test_x)
            
            (rmse,mae,r2)=self.eval_metrics(test_y,predicted_qualities)
            
            scores={"rmse":rmse,"mae":mae,"r2":r2}
            save_json(path=Path(self.config.metric_file_name),data=scores)
            
            mlflow.log_params(self.config.all_params)
            
            mlflow.log_metric("rmse",rmse)
            mlflow.log_metric("r2",r2)
            mlflow.log_metric("mae",mae)
            
            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, "model")
            
        
        
        

In [28]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-06-25 23:34:58,502 : INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-25 23:34:58,512 : INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-25 23:34:58,514 : INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-25 23:34:58,515 : INFO: common: created directory at: artifacts]
[2026-06-25 23:34:58,515 : INFO: common: created directory at: artifacts/model_evaluation]
[2026-06-25 23:34:59,717 : INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]


2026/06/25 23:35:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/25 23:35:01 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in d:\Revision\End-to-End-ML-project
2026/06/25 23:35:04 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in d:\Revision\End-to-End-ML-project
2026/06/25 23:35:04 INFO mlflow.utils.environment: Detected uv project at d:\Revision\End-to-End-ML-project. Attempting to export requirements via 'uv export'.
2026/06/25 23:35:04 INFO mlflow.utils.uv_utils: Exported 119 dependencies via uv
2026/06/25 23:35:04 INFO mlflow.utils.environment: Successfully exported 119 requirements from uv project. Skipping package capture based inference.
2026/06/25 23:35:04 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'ElasticnetModel' already exist

🏃 View run indecisive-elk-503 at: https://dagshub.com/shu7620/End-to-End-ML-project.mlflow/#/experiments/0/runs/6ad1e3163dd847c4a61547e824787570
🧪 View experiment at: https://dagshub.com/shu7620/End-to-End-ML-project.mlflow/#/experiments/0
